# quasi_newton

L?utilisateur fournit la fonction objectif et le point initial. Les derivees sont estimees numeriquement. La fonction retourne aussi un message explicite en cas de non-convergence ou de blocage numerique; la derniere cellule donne un exemple.


In [ ]:
import numpy as np


def quasi_newton(f, x0, tol=1e-5, max_iter=100, alpha0=1.0,
                 beta=0.5, c1=1e-4):
    """Minimise f par BFGS avec gradient numerique.

    Retourne (x_opt, f_opt, n_iter, converged, message). Les echecs de
    converged vaut True uniquement si la norme du gradient est <= tol. Il vaut
    False si le maximum d'iterations est atteint ou si l'algorithme doit s'arreter;
    le message precise la raison, sans que cela soit necessairement une erreur Python.
    """
    x = np.asarray(x0, dtype=float)
    if x.ndim != 1 or x.size == 0 or not np.all(np.isfinite(x)):
        raise ValueError("x0 doit etre un vecteur non vide de valeurs finies.")
    if tol <= 0 or max_iter < 0 or alpha0 <= 0:
        raise ValueError("tol et alpha0 doivent etre positifs, max_iter positif ou nul.")
    if not 0 < beta < 1 or not 0 < c1 < 1:
        raise ValueError("beta et c1 doivent etre compris entre 0 et 1.")

    # Les fonctions a croissance rapide peuvent deborder pendant les essais de pas.
    def evaluer_f(point):
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            return float(f(point))

    def produit_scalaire(a, b):
        with np.errstate(over="ignore", invalid="ignore"):
            return float(np.dot(a, b))

    def norme_vecteur(vecteur):
        return float(np.hypot.reduce(vecteur))

    # Estime chaque derivee partielle par differences finies centrees.
    def gradient_numerique(point):
        gradient = np.empty_like(point)
        pas_base = np.cbrt(np.finfo(float).eps)
        for i in range(point.size):
            pas = pas_base * max(1.0, abs(point[i]))
            point_plus = point.copy()
            point_moins = point.copy()
            point_plus[i] += pas
            point_moins[i] -= pas
            valeur_plus = evaluer_f(point_plus)
            valeur_moins = evaluer_f(point_moins)
            if not np.isfinite(valeur_plus) or not np.isfinite(valeur_moins):
                return None
            gradient[i] = (valeur_plus - valeur_moins) / (2 * pas)
        if not np.all(np.isfinite(gradient)):
            return None
        return gradient
    try:
        valeur = evaluer_f(x)
    except (TypeError, ValueError, OverflowError) as erreur:
        message = f"Impossible d'evaluer f(x0): {erreur}. Verifiez la dimension de x0 et la definition de f."
        return x, np.nan, 0, False, message
    if not np.isfinite(valeur):
        return x, valeur, 0, False, "La fonction n'est pas finie au point initial."
    gradient = gradient_numerique(x)
    if gradient is None:
        return x, valeur, 0, False, "Le gradient numerique n'a pas pu etre evalue."

    inverse_hessienne = np.eye(x.size)
    for iteration in range(max_iter):
        if norme_vecteur(gradient) <= tol:
            return x, valeur, iteration, True, "Convergence atteinte."

        direction = -inverse_hessienne @ gradient
        if produit_scalaire(gradient, direction) >= 0:
            inverse_hessienne = np.eye(x.size)
            direction = -gradient

        pente = produit_scalaire(gradient, direction)
        alpha = alpha0
        for _ in range(60):
            with np.errstate(over="ignore", invalid="ignore"):
                x_suivant = x + alpha * direction
            if np.all(np.isfinite(x_suivant)):
                valeur_suivante = evaluer_f(x_suivant)
                if np.isfinite(valeur_suivante) and valeur_suivante <= valeur + c1 * alpha * pente:
                    break
            alpha *= beta
        else:
            return x, valeur, iteration, False, "Aucun pas n'a permis une diminution suffisante de f. Arret sans convergence; verifiez que f admet un minimum et le choix des parametres."

        gradient_suivant = gradient_numerique(x_suivant)
        if gradient_suivant is None:
            return x_suivant, valeur_suivante, iteration + 1, False, "Gradient numerique non fini; arret de l'algorithme."

        s = x_suivant - x
        y = gradient_suivant - gradient
        courbure = produit_scalaire(y, s)
        if np.isfinite(courbure) and courbure > 1e-12 * norme_vecteur(y) * norme_vecteur(s):
            rho = 1.0 / courbure
            identite = np.eye(x.size)
            facteur = identite - rho * np.outer(s, y)
            inverse_hessienne = facteur @ inverse_hessienne @ facteur.T + rho * np.outer(s, s)

        x, valeur, gradient = x_suivant, valeur_suivante, gradient_suivant

    if norme_vecteur(gradient) <= tol:
        return x, valeur, max_iter, True, "Convergence atteinte."
    return x, valeur, max_iter, False, "Nombre maximal d'iterations atteint sans convergence."


In [ ]:
# Exemple : minimisation de la fonction de l'exercice du cours

def f(X):
    x, y, z = X
    return 6*x**2 + 6*x*y + 3*y**2 + y**3 - (1/3)*z**3 + (1/4)*z**4 


x0 = np.array([-0.6637, 0.1758, 1.5778])
x_opt, f_opt, n_iter, converged, message = quasi_newton(
    f, x0, tol=1e-5, max_iter=100
)

print("Point obtenu :", x_opt)
print("Valeur de la fonction :", f_opt)
print("Nombre d'iterations :", n_iter)
print("Convergence :", converged)
print("Message :", message)
